In [1]:
import numpy as np

import skimage as ski

In [2]:
img = ski.data.coins()

In [3]:
Hrr, Hrc, Hcc = ski.feature.hessian_matrix(img,
                                           sigma=1,
                                           order='rc',
                                           use_gaussian_derivatives=True)

/Volumes/zorg/mb312/dev_trees/coordinate-review/main/src/skimage/feature/peak.py:10: ExperimentalAPIWarning: Importing from the `skimage2` namespace is experimental. Its API is under development and considered unstable!
  import skimage2 as ski2


In [4]:
Hxx, Hxy, Hyy = ski.feature.hessian_matrix(img,
                                           sigma=1,
                                           order='xy',
                                           use_gaussian_derivatives=True)

In [5]:
[np.max(np.abs(a)) for a in (Hxx, Hxy, Hyy)]

[np.float64(0.20859111547637055),
 np.float64(0.12339236135557885),
 np.float64(0.2089624932462676)]

In [6]:
def maxabsd(a1, a2):
    return np.max(np.abs(a1 - a2))

In [7]:
maxabsd(Hrr, Hyy), maxabsd(Hrc, Hxy), maxabsd(Hcc, Hxx)

(np.float64(0.0), np.float64(0.02431522983711614), np.float64(0.0))

In [8]:
from functools import partial

# Future incantantion.
hessian_matrix = partial(ski.feature.hessian_matrix,
                         use_gaussian_derivatives=True)

def rc_xy_d(func, img, sigma=0.1):
    Hrr, Hrc, Hcc = func(img, sigma, order='rc')
    Hxx, Hxy, Hyy = func(img, sigma, order='xy')
    return maxabsd(Hrr, Hyy), maxabsd(Hrc, Hxy), maxabsd(Hcc, Hxx)

In [9]:
rc_xy_d(hessian_matrix, img, 5)

(np.float64(0.0), np.float64(0.0012710362074207595), np.float64(0.0))

In [10]:
rc_xy_d(ski.feature.structure_tensor, img)

(np.float64(0.0), np.float64(0.0), np.float64(0.0))

In [11]:
def get_xy_from_rc(func, image, **kwargs):
    """Replicate order='xy' using order='rc' logic.
   
    Gemini function.
    """
    # Transpose image to reverse axes
    image_t = np.transpose(image, axes=range(image.ndim)[::-1])
    
    # Handle sigma if provided as a sequence
    if 'sigma' in kwargs and not np.isscalar(kwargs['sigma']):
        kwargs['sigma'] = kwargs['sigma'][::-1]
        
    # Call the 'rc' version
    res_rc = func(image_t, order='rc', **kwargs)
    
    # Transpose elements back
    return [np.transpose(h, axes=range(image.ndim)[::-1]) for h in res_rc]

In [12]:
# Replicate 'xy' order with Gemini wrapper.
Hxx_g, Hxy_g, Hyy_g = get_xy_from_rc(hessian_matrix, img, sigma=1)

In [13]:
maxabsd(Hxx, Hxx_g), maxabsd(Hxy, Hxy_g), maxabsd(Hyy, Hyy_g)

(np.float64(1.1102230246251565e-16),
 np.float64(6.938893903907228e-17),
 np.float64(8.326672684688674e-17))

In [14]:
# Now without use_gaussian_derivatives future argument.
Hxx_g2, Hxy_g2, Hyy_g2 = get_xy_from_rc(ski.feature.hessian_matrix, img, sigma=1)

/var/folders/hd/rfxyn9gx4bl39bvwzrgn3rtr0000gn/T/ipykernel_22155/774630541.py:14: FutureWarning: use_gaussian_derivatives currently defaults to False, but will change to True in a future version. Please specify this argument explicitly to maintain the current behavior
  res_rc = func(image_t, order='rc', **kwargs)


In [15]:
maxabsd(Hxx, Hxx_g2), maxabsd(Hxy, Hxy_g2), maxabsd(Hyy, Hyy_g2)

(np.float64(0.09669888839943355),
 np.float64(0.03937443629353066),
 np.float64(0.07836800752352306))